# Dai en vivo: servidor para la demo de PocketPal

Este cuaderno prende a **Dai** en Colab y le da una **dirección web pública**. Con esa dirección:
- **La app de Android** le envía la foto de la factura y recibe los 4 campos (monto, fecha, categoría y motivo).
- **Plan B:** abriendo `…/prueba` en el navegador del celular se puede tomar la foto y ver el resultado, sin la app.

Dai usa exactamente lo que se evaluó en la comparación:
| Campo | Cómo lo obtiene |
|---|---|
| Texto de la foto | PaddleOCR (modo **rápido** = modelos *mobile*, pensados para celulares) |
| Monto y fecha | Las mismas reglas del cuaderno de comparación |
| Categoría | TF-IDF + Regresión logística, entrenada con las facturas etiquetadas |
| Motivo | k-NN (k = 1): el motivo de la factura más parecida, máximo 50 caracteres |

**Requisito:** haber corrido `comparacion_ocr_facturas_v2` con Drive activado (usa `facturas.csv` y `textos_ocr.json` que quedaron en tu Drive).

**Pasos:** corre las celdas en orden. La primera vez la instalación tarda unos minutos. Deja la pestaña abierta durante toda la demo: si Colab se cierra, la dirección deja de funcionar.

In [ ]:
# ===================== CONFIGURACIÓN =====================
MODO_OCR = "rapido"     # "rapido" = PaddleOCR mobile (pocos segundos) | "completo" = el mismo modelo de la comparación (más lento)
LADO_MAX = 1600         # la foto se reduce a este lado máximo antes de leerla (más pequeña = más rápida)
TRABAJO = "/content/drive/MyDrive/comparacion_ocr_facturas"   # carpeta que creó el cuaderno de comparación

import os, sys, json, time, glob, subprocess, re
from google.colab import drive
drive.mount("/content/drive")

def buscar(nombre, raiz):
    for r, _, fs in os.walk(raiz):
        if nombre in fs:
            return os.path.join(r, nombre)
    return None

CSV = buscar("facturas.csv", os.path.join(TRABAJO, "dataset"))
TEXTOS = os.path.join(TRABAJO, "resultados", "textos_ocr.json")
assert CSV, "No encontré facturas.csv. Corre primero el cuaderno de comparación con GUARDAR_EN_DRIVE = True."
assert os.path.exists(TEXTOS), "No encontré textos_ocr.json. Corre la celda 12 (Guardar) del cuaderno de comparación."
assert "paddleocr" in json.load(open(TEXTOS, encoding="utf-8")), "textos_ocr.json no trae los textos de PaddleOCR."
print("Dataset:", CSV)
print("Textos:", TEXTOS)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/comparacion_ocr_facturas/dataset/facturas.csv
Textos: /content/drive/MyDrive/comparacion_ocr_facturas/resultados/textos_ocr.json


## 1. Instalar (solo la primera vez en cada sesión)
PaddleOCR se instala en un entorno aparte, igual que en la comparación, para que no choque con Colab.

In [ ]:
ENV = "/content/dai_env"
PY = f"{ENV}/bin/python"

def sh(args):
    r = subprocess.run(args, capture_output=True, text=True)
    return r.returncode, (r.stdout or "") + (r.stderr or "")

if not os.path.exists(f"{ENV}/.listo"):
    sh([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
    c, out = sh([sys.executable, "-m", "virtualenv", "-q", ENV])
    assert c == 0, out[-800:]
    pip = [PY, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    print("Instalando PaddleOCR (unos minutos)...")
    c, out = sh(pip + ["paddlepaddle==3.2.2", "paddleocr"])
    if c != 0:
        c, out = sh(pip + ["paddlepaddle", "paddleocr"])
    assert c == 0, out[-1500:]
    print("Instalando el servidor web y scikit-learn...")
    c, out = sh(pip + ["fastapi", "uvicorn", "python-multipart", "scikit-learn", "pandas", "pillow", "pillow-heif"])
    assert c == 0, out[-1500:]
    open(f"{ENV}/.listo", "w").close()
else:
    print("Entorno ya instalado.")

if not os.path.exists("/content/cloudflared"):
    c, out = sh(["wget", "-q", "-O", "/content/cloudflared",
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    assert c == 0, out
    os.chmod("/content/cloudflared", 0o755)
print("Listo.")

Instalando PaddleOCR (unos minutos)...
Instalando el servidor web y scikit-learn...
Listo.


## 2. El programa de Dai
Esta celda escribe el archivo del servidor. No hay que cambiarle nada.

In [ ]:
%%writefile /content/dai_servidor.py
# -*- coding: utf-8 -*-
"""
Servidor de Dai (PocketPal): recibe la foto de una factura y responde los 4 campos.
  - OCR: PaddleOCR (el modelo que ganó la comparación)
  - Monto y fecha: las mismas reglas del cuaderno de comparación
  - Categoría: TF-IDF + Regresión logística, entrenada con las facturas etiquetadas
  - Motivo: k-NN (k = 1), copia el motivo de la factura más parecida (máximo 50 caracteres)
Rutas:
  GET  /          -> estado del servidor
  GET  /prueba    -> página web para probar desde el celular (plan B de la demo)
  POST /analizar  -> recibe la foto (campo "foto") y devuelve JSON
"""
import os, re, io, json, time, tempfile, unicodedata
import datetime as dt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
try:
    import pillow_heif
    pillow_heif.register_heif_opener()      # fotos de iPhone (.heic)
except Exception:
    pass
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline

CSV = os.environ["DAI_CSV"]                  # facturas.csv del dataset limpio
TEXTOS = os.environ["DAI_TEXTOS"]            # textos_ocr.json que guardó el cuaderno de comparación
MODO = os.environ.get("DAI_MODO", "rapido")  # "rapido" = PaddleOCR mobile | "completo" = el de la comparación
LADO = int(os.environ.get("DAI_LADO", "1600"))
LIMITE_MOTIVO = 50

# ============================== REGLAS DE MONTO Y FECHA (iguales al cuaderno) ==============================
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def a_monto(v):
    # 2 dígitos tras el último separador = centavos; 3 dígitos = miles
    s = re.sub(r"[^\d.,]", "", str(v))
    if not re.search(r"\d", s):
        return None
    if "." in s and "," in s:
        dec = "." if s.rfind(".") > s.rfind(",") else ","
    elif "." in s or "," in s:
        sep = "." if "." in s else ","
        dec = sep if (s.count(sep) == 1 and len(s.split(sep)[-1]) in (1, 2)) else None
    else:
        dec = None
    if dec:
        ent, _, frac = s.rpartition(dec)
        return round(float(f"{re.sub(r'[^0-9]', '', ent) or 0}.{re.sub(r'[^0-9]', '', frac)}"), 2)
    return float(re.sub(r"[^0-9]", "", s))

MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7,
         "ago": 8, "sep": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12}
RE_F1 = re.compile(r"(?<!\d)(\d{4})\s*[/\-.]\s*(\d{1,2})\s*[/\-.]\s*(\d{1,2})(?!\d)")
RE_F2 = re.compile(r"(?<!\d)(\d{1,2})\s*[/\-.]\s*(\d{1,2})\s*[/\-.]\s*(\d{4}|\d{2})(?!\d)")
RE_F3 = re.compile(r"(?<!\d)(\d{1,2})[\s\-/]*(?:de\s+)?([A-Za-zÀ-ÿ]{3,10})\.?,?[\s\-/]*(?:de\s+|del\s+)?(\d{4})(?!\d)", re.I)   # 8 oct, 2026 / 08 Sep. 2026 / 23-Sep-2026
RE_F4 = re.compile(r"(?<![A-Za-z])([A-Za-zÀ-ÿ]{3})\.?\s*/\s*(\d{1,2})\s*/\s*(\d{4})(?!\d)", re.I)   # OCT/02/2026 (recibos del Acueducto)

def fechas_en(linea):
    sal = []
    def agregar(y, mo, d):
        try:
            if 2000 <= y <= 2100:
                dt.date(y, mo, d)
                sal.append(f"{y:04d}-{mo:02d}-{d:02d}")
        except ValueError:
            pass
    for m in RE_F1.finditer(linea):
        agregar(int(m[1]), int(m[2]), int(m[3]))
    for m in RE_F2.finditer(linea):
        y = int(m[3])
        agregar(y + 2000 if y < 100 else y, int(m[2]), int(m[1]))
    for m in RE_F3.finditer(linea):
        mes = MESES.get(sin_tildes(m[2]).lower()[:3])
        if mes:
            agregar(int(m[3]), mes, int(m[1]))
    for m in RE_F4.finditer(linea):
        mes = MESES.get(sin_tildes(m[1]).lower()[:3])
        if mes:
            agregar(int(m[3]), mes, int(m[2]))
    return sal

def elegir_fecha(texto):
    lineas = [re.sub(r"(\d)\s*([/\-.])\s*(\d)", r"\1\2\3", l) for l in texto.splitlines()]
    cand = [f for l in lineas for f in fechas_en(l)]
    if not cand:
        return None, set()
    for patron in (r"oportuno", r"fecha|emisi|pago"):
        cerca = []
        for i, l in enumerate(lineas):
            if re.search(patron, l, re.I):
                for j in (i, i + 1, i + 2):      # a veces la fecha queda en la línea siguiente
                    if j < len(lineas):
                        cerca += fechas_en(lineas[j])
        if cerca:
            return cerca[0], set(cand)
    return cand[0], set(cand)

def normalizar(linea):
    # Corrige errores típicos del OCR en las cifras, igual para los cuatro modelos:
    # "85 . 000" -> "85.000", "249 900" -> "249.900", "1 234 567" -> "1.234.567",
    # "1l.000" -> "11.000", "4O.500" -> "40.500"
    linea = re.sub(r"([\dlIoO])\s*([.,])\s*(\d)", r"\1\2\3", linea)
    def arreglar_token(m):
        t = m.group(0)
        if sum(ch.isdigit() for ch in t) >= 2:
            t = t.translate(str.maketrans({"l": "1", "I": "1", "|": "1", "o": "0", "O": "0"}))
        return t
    linea = re.sub(r"(?<![A-Za-z])[\dlIoO|][\dlIoO|.,]*\d[\dlIoO|.,]*", arreglar_token, linea)
    linea = re.sub(r"(?<![\d.,])\d{1,3}(?:\s\d{3})+(?![\d.,]?\d)",
                   lambda m: m.group(0).replace(" ", "."), linea)
    return linea

RE_MONTO = re.compile(r"(?<![\d.,])(\d{1,3}(?:[.,]\d{3})+(?:[.,]\d{1,2})?|\d+[.,]\d{1,2}|\d{4,7})(?!\d)")
IGNORAR = re.compile(r"\bnit\b|cufe|resoluc|autoriz|tel[eé]f|\btel\b|\bcel\b|celular|whatsapp|n[uú]mero|prefijo|pedido|referencia|cuenta|c[eé]dula|rango|vigencia", re.I)
GRUPOS = [r"total\s*a\s*pagar|valor\s*a\s*pagar|neto\s*a\s*pagar|total\s*a\s*cancelar|pago\s*total",
          r"total\s*factura|valor\s*total|total\s*venta|total\s*compra|total\s*neto|total\s*general",
          r"(?<!sub)total(?!\s*(?:iva|impuesto|descuento|art|[ií]tem|l[ií]neas|unid|cant|bruto))"]

def montos_en(linea, cero=False):
    for rx in (RE_F1, RE_F2, RE_F3, RE_F4):
        linea = rx.sub(" ", linea)
    vals = []
    for m in RE_MONTO.finditer(linea):
        v = a_monto(m[1])
        if v is not None and 100 <= v < 50_000_000:
            vals.append(v)
    # Un total de $0 (por ejemplo, una factura de celular ya cubierta por un saldo a favor)
    # solo se acepta en la línea "TOTAL A PAGAR", para no confundirlo con cualquier cero suelto
    if cero and not vals and re.search(r"\$\s*0(?:[.,]00?)?(?![\d.,]*\d)", linea):
        vals.append(0.0)
    return vals

def elegir_monto(texto):
    lineas = [normalizar(l) for l in texto.splitlines()]
    todos = {v for l in lineas for v in montos_en(l)}
    utiles = [i for i, l in enumerate(lineas) if not IGNORAR.search(l)]
    for k, g in enumerate(GRUPOS):
        cero = (k == 0)      # solo "TOTAL A PAGAR" y similares pueden valer $0
        halladas = []
        for i in utiles:
            if re.search(g, lineas[i], re.I):
                vs = montos_en(lineas[i], cero)
                if not vs:    # a veces el OCR deja la cifra en la línea siguiente
                    vs = [v for j in (i + 1, i + 2) if j < len(lineas) for v in montos_en(lineas[j], cero)]
                halladas += vs
        if halladas:
            return max(halladas), todos | set(halladas)
    resto = [v for i in utiles for v in montos_en(lineas[i])]
    return (max(resto) if resto else None), todos

# ============================== ENTRENAMIENTO (categoría y motivo) ==============================
df = pd.read_csv(CSV)
df["categoria"] = (df["categoria"].fillna("Sin categoría").astype(str).str.strip()
                   .str.replace(r"\s+", " ", regex=True).str.capitalize())
with open(TEXTOS, encoding="utf-8") as f:
    textos_guardados = json.load(f)
X = [str(t) for t in textos_guardados["paddleocr"]]
assert len(X) == len(df), f"textos_ocr.json tiene {len(X)} textos y facturas.csv {len(df)} filas: no coinciden"
y = df["categoria"].values
motivos = df["motivo"].fillna("").astype(str).str.strip().values

def tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), strip_accents="unicode",
                           lowercase=True, sublinear_tf=True)

# Para la app se entrena con TODAS las facturas: la evaluación ya se hizo en el cuaderno con la separación 75/25.
clasificador = make_pipeline(tfidf(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(X, y)
vec_motivo = tfidf().fit(X)
vecino = NearestNeighbors(n_neighbors=1, metric="cosine").fit(vec_motivo.transform(X))
print(f"Modelos entrenados con {len(X)} facturas y {len(set(y))} categorías", flush=True)

# ============================== OCR ==============================
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
from paddleocr import PaddleOCR

def crear_lector(mkldnn):
    comunes = dict(use_doc_orientation_classify=False, use_doc_unwarping=False,
                   use_textline_orientation=True, device="cpu", enable_mkldnn=mkldnn)
    if MODO == "rapido":
        # Modelos "mobile": los pensados para celulares, mucho más rápidos en CPU
        return PaddleOCR(text_detection_model_name="PP-OCRv5_mobile_det",
                         text_recognition_model_name="latin_PP-OCRv5_mobile_rec", **comunes)
    return PaddleOCR(lang="es", **comunes)

def agrupar_lineas(cajas):
    cajas = sorted([c for c in cajas if str(c[4]).strip()], key=lambda c: (c[1] + c[3]) / 2)
    lineas = []
    for c in cajas:
        cy, h = (c[1] + c[3]) / 2, max(1.0, c[3] - c[1])
        if lineas and abs(cy - lineas[-1]["cy"]) < 0.5 * max(h, lineas[-1]["h"]):
            L = lineas[-1]
            L["cajas"].append(c)
            L["cy"] = sum((k[1] + k[3]) / 2 for k in L["cajas"]) / len(L["cajas"])
            L["h"] = max(L["h"], h)
        else:
            lineas.append({"cy": cy, "h": h, "cajas": [c]})
    return "\n".join(" ".join(str(k[4]) for k in sorted(L["cajas"], key=lambda k: k[0])) for L in lineas)

def leer(ruta):
    cajas = []
    for r in lector.predict(ruta):
        textos = list(r["rec_texts"])
        try:
            boxes = [list(map(float, b)) for b in r["rec_boxes"]]
        except Exception:
            boxes = [[min(q[0] for q in pl), min(q[1] for q in pl), max(q[0] for q in pl), max(q[1] for q in pl)]
                     for pl in r["rec_polys"]]
        cajas += [(b[0], b[1], b[2], b[3], t) for b, t in zip(boxes, textos)]
    return agrupar_lineas(cajas)

def preparar(contenido):
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(contenido))).convert("RGB")
    if max(img.size) > LADO:
        img.thumbnail((LADO, LADO))
    tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
    img.save(tmp.name, quality=92)
    return tmp.name

# Imagen de calentamiento: la primera lectura siempre es la más lenta
_calienta = Image.new("RGB", (800, 400), (255, 255, 255))
from PIL import ImageDraw
ImageDraw.Draw(_calienta).text((40, 180), "TOTAL A PAGAR $ 45.000  FECHA 10/11/2026", fill=(0, 0, 0))
_ruta_cal = os.path.join(tempfile.gettempdir(), "dai_calentamiento.jpg")
_calienta.save(_ruta_cal)

t0 = time.time()
lector = crear_lector(True)
try:
    leer(_ruta_cal)
except Exception as e:
    print("oneDNN falló, se reintenta sin oneDNN:", str(e)[:120], flush=True)
    lector = crear_lector(False)
    leer(_ruta_cal)
print(f"PaddleOCR listo en modo '{MODO}' ({time.time() - t0:.1f} s)", flush=True)

# ============================== DAI ==============================
NO_COMERCIO = re.compile(r"nit|factura|fecha|tel|cel|direcci|cufe|resoluc|www|http|@|r[eé]gimen|iva|cliente", re.I)

def comercio_en(texto):
    # El nombre del comercio suele ser la primera línea con letras que no es un dato del documento
    for linea in texto.splitlines()[:8]:
        limpia = linea.strip(" -:*.")
        if len(re.findall(r"[A-Za-zÁÉÍÓÚÑáéíóúñ]", limpia)) >= 3 and not NO_COMERCIO.search(limpia):
            return limpia[:40]
    return ""

def analizar_texto(texto):
    monto, _ = elegir_monto(texto)
    fecha, _ = elegir_fecha(texto)
    proba = clasificador.predict_proba([texto])[0]
    k = int(np.argmax(proba))
    _, idx = vecino.kneighbors(vec_motivo.transform([texto]))
    motivo = motivos[idx[0][0]][:LIMITE_MOTIVO]
    return {
        "monto": None if monto is None else round(float(monto), 2),
        "fecha": fecha,
        "categoria": str(clasificador.classes_[k]),
        "confianza_categoria": round(float(proba[k]), 2),
        "motivo": motivo,
        "comercio": comercio_en(texto),
    }

app = FastAPI(title="Dai · PocketPal")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def estado():
    return {"estado": "listo", "modo_ocr": MODO, "facturas_entrenamiento": len(X)}

@app.post("/analizar")
async def analizar(foto: UploadFile = File(...)):
    t = time.time()
    contenido = await foto.read()
    try:
        ruta = preparar(contenido)
    except Exception:
        return JSONResponse({"error": "No pude abrir la imagen. Envía una foto JPG o PNG."}, status_code=400)
    try:
        t_ocr = time.time()
        texto = leer(ruta)
        seg_ocr = time.time() - t_ocr
    finally:
        os.remove(ruta)
    salida = analizar_texto(texto)
    salida["segundos"] = round(time.time() - t, 2)
    salida["segundos_ocr"] = round(seg_ocr, 2)
    salida["texto_leido"] = texto[:3000]
    print(f"Factura analizada en {salida['segundos']} s: {salida['monto']} | {salida['fecha']} | "
          f"{salida['categoria']} | {salida['motivo']}", flush=True)
    return salida

PAGINA = """<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Dai · Prueba</title>
<style>
body{margin:0;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:#FBF7F6;color:#2A0609}
header{background:#5C0F14;color:#FBF7F6;padding:20px 16px}h1{margin:0;font-size:22px}
main{max-width:520px;margin:0 auto;padding:16px}
label.boton{display:block;text-align:center;background:#A3232A;color:#fff;padding:16px;border-radius:999px;font-weight:700;font-size:18px;cursor:pointer}
input{display:none}#vista{width:100%;border-radius:12px;margin-top:14px;display:none}
.card{background:#fff;border-radius:16px;padding:16px;margin-top:16px;box-shadow:0 6px 18px rgba(92,15,20,.08)}
.fila{display:flex;justify-content:space-between;padding:10px 0;border-bottom:1px solid #EFE7E6;font-size:18px}
.fila:last-child{border-bottom:none}.fila span{color:#7A6A69}.fila b{text-align:right;max-width:60%}
#estado{margin-top:14px;font-size:16px;color:#7A6A69;text-align:center}
.confirmar{display:block;width:100%;margin-top:14px;background:#E0A030;border:none;padding:14px;border-radius:999px;font-weight:700;font-size:18px}
</style></head><body>
<header><h1>Dai · Escanear factura</h1></header><main>
<label class="boton">📷 Tomar o escoger foto<input id="foto" type="file" accept="image/*" capture="environment"></label>
<img id="vista" alt="Factura">
<div id="estado"></div>
<div class="card" id="res" style="display:none">
<div class="fila"><span>Monto</span><b id="monto"></b></div>
<div class="fila"><span>Fecha</span><b id="fecha"></b></div>
<div class="fila"><span>Categoría</span><b id="categoria"></b></div>
<div class="fila"><span>Motivo</span><b id="motivo"></b></div>
<button class="confirmar" type="button">Confirmar</button>
</div></main>
<script>
const $ = id => document.getElementById(id);
$("foto").addEventListener("change", async e => {
  const f = e.target.files[0]; if (!f) return;
  $("vista").src = URL.createObjectURL(f); $("vista").style.display = "block";
  $("res").style.display = "none"; $("estado").textContent = "Dai está leyendo la factura…";
  const datos = new FormData(); datos.append("foto", f);
  try {
    const r = await fetch("analizar", {method: "POST", body: datos});
    const j = await r.json();
    if (!r.ok) throw new Error(j.error || r.status);
    $("monto").textContent = j.monto == null ? "—" : "$" + Math.round(j.monto).toLocaleString("es-CO");
    $("fecha").textContent = j.fecha ? j.fecha.split("-").reverse().join("/") : "—";
    $("categoria").textContent = j.categoria;
    $("motivo").textContent = j.motivo || "—";
    $("res").style.display = "block";
    $("estado").textContent = "Listo en " + j.segundos + " s";
  } catch (err) { $("estado").textContent = "No se pudo analizar: " + err.message; }
});
</script></body></html>"""

@app.get("/prueba", response_class=HTMLResponse)
def prueba():
    return PAGINA

Writing /content/dai_servidor.py


## 3. Prender a Dai y obtener la dirección pública
La primera vez descarga los modelos de PaddleOCR (1 o 2 minutos). Al final imprime **dos direcciones**:
- la de la **app** (pégala en PocketPal),
- la de **prueba** (ábrela en el navegador del celular).

La dirección cambia cada vez que corres esta celda.

In [ ]:
import requests

# Apaga una versión anterior si la había
for p in ("uvicorn dai_servidor", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", p])
time.sleep(2)

entorno = dict(os.environ, DAI_CSV=CSV, DAI_TEXTOS=TEXTOS, DAI_MODO=MODO_OCR, DAI_LADO=str(LADO_MAX),
               PYTHONUNBUFFERED="1")
log_srv = open("/content/dai_servidor.log", "w")
servidor = subprocess.Popen([PY, "-m", "uvicorn", "dai_servidor:app", "--host", "0.0.0.0", "--port", "8000"],
                            cwd="/content", env=entorno, stdout=log_srv, stderr=subprocess.STDOUT)

print("Prendiendo a Dai", end="", flush=True)
inicio = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/", timeout=2).ok:
            break
    except Exception:
        pass
    if servidor.poll() is not None:
        print("\nEl servidor se detuvo. Últimas líneas del registro:")
        print(open("/content/dai_servidor.log").read()[-2500:])
        raise SystemExit
    if time.time() - inicio > 900:
        raise SystemExit("Tardó demasiado. Revisa /content/dai_servidor.log")
    print(".", end="", flush=True)
    time.sleep(5)
print(f"\nDai está listo ({time.time() - inicio:.0f} s).")
print("\n".join(l for l in open("/content/dai_servidor.log").read().splitlines() if "listo" in l or "entrenados" in l))

log_tun = open("/content/tunel.log", "w")
tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                         stdout=log_tun, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunel.log").read())
    if m:
        URL = m.group(0)
        break
assert URL, "No se pudo crear la dirección pública. Vuelve a correr esta celda."
time.sleep(5)
print("\n==============================================")
print("Dirección para la app:   ", URL)
print("Página de prueba (celular):", URL + "/prueba")
print("==============================================")

Prendiendo a Dai....
Dai está listo (20 s).
Modelos entrenados con 143 facturas y 10 categorías
PaddleOCR listo en modo 'rapido' (3.4 s)

Dirección para la app:    https://impression-unlock-written-beyond.trycloudflare.com
Página de prueba (celular): https://impression-unlock-written-beyond.trycloudflare.com/prueba


## 4. Prueba de velocidad y aciertos (antes de la demo)
Envía algunas facturas del dataset a Dai, tal como lo haría la app, y muestra el tiempo y lo que respondió frente al valor real.

In [ ]:
import pandas as pd
N_PRUEBA = 8
df = pd.read_csv(CSV)
carpeta = os.path.dirname(CSV)
muestra = df.sample(N_PRUEBA, random_state=7)
filas = []
for _, f in muestra.iterrows():
    ruta = os.path.join(carpeta, "imagenes", str(f["foto"]).split(";")[0])
    with open(ruta, "rb") as fh:
        r = requests.post("http://localhost:8000/analizar", files={"foto": fh}, timeout=300).json()
    filas.append({"foto": os.path.basename(ruta),
                  "monto_real": f["monto_cop"], "monto_dai": r["monto"],
                  "fecha_real": f["fecha"], "fecha_dai": r["fecha"],
                  "categoria_real": f["categoria"], "categoria_dai": r["categoria"],
                  "motivo_dai": r["motivo"], "segundos": r["segundos"]})
prueba = pd.DataFrame(filas)
display(prueba)
print(f"Tiempo promedio por factura: {prueba['segundos'].mean():.1f} s (modo {MODO_OCR})")
print("Ojo: estas facturas también se usaron para entrenar la app, así que no es una evaluación;")
print("la evaluación válida es la del cuaderno de comparación. Esta celda sirve para medir el tiempo.")

,foto,monto_real,monto_dai,fecha_real,fecha_dai,categoria_real,categoria_dai,motivo_dai,segundos
0,factura_012_1.jpeg,46300.0,46300.0,2026-10-26,2026-10-26,Restaurantes y domicilios,Restaurantes y domicilios,Hamburguesa,21.52
1,factura_122_1.png,10000.0,1895000.0,2026-10-14,2026-04-22,Transporte,Transporte,gasolina,31.15
2,factura_038_1.jpeg,20000.0,20000.0,2024-06-29,2016-11-16,Supermercado,Supermercado,Gastos personales,7.13
3,factura_033_1.jpeg,110000.0,110000.0,2025-12-20,2025-12-20,Supermercado,Supermercado,gastos personales,11.42
4,factura_071_1.jpg,37508.0,657802.0,2026-10-20,2026-10-20,Servicios,Servicios,Factura del agua y alcantarillado de octubre,9.86
5,factura_119_1.png,10000.0,1886267.0,2026-10-02,2026-04-22,Transporte,Transporte,Gasolina,14.40
6,factura_078_1.jpg,53900.0,53900.0,2026-09-04,2026-09-04,Servicios,Servicios,Factura Movistar de septiembre,9.51
7,factura_116_1.jpg,15030.0,15030.0,2026-10-21,2026-10-21,Transporte,Transporte,Pon tu precio - Didi,6.38


Tiempo promedio por factura: 13.9 s (modo rapido)
Ojo: estas facturas también se usaron para entrenar la app, así que no es una evaluación;
la evaluación válida es la del cuaderno de comparación. Esta celda sirve para medir el tiempo.


## 5. Ver lo que está pasando (opcional)
Muestra las últimas facturas que llegaron desde la app o desde la página de prueba.

In [ ]:
print("\n".join(open("/content/dai_servidor.log").read().splitlines()[-15:]))

Factura analizada en 11.42 s: 110000.0 | 2025-12-20 | Supermercado | gastos personales
INFO:     127.0.0.1:53538 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 9.86 s: 657802.0 | 2026-10-20 | Servicios | Factura del agua y alcantarillado de octubre
INFO:     127.0.0.1:45630 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 14.4 s: 1886267.0 | 2026-04-22 | Transporte | Gasolina
INFO:     127.0.0.1:59468 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 9.18 s: 7200.0 | 2026-11-04 | Restaurantes y domicilios | Compra de desayuno
INFO:     2800:484:3840:3900:e560:b836:72d6:c649:0 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 9.51 s: 53900.0 | 2026-09-04 | Servicios | Factura Movistar de septiembre
INFO:     127.0.0.1:45402 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 6.38 s: 15030.0 | 2026-10-21 | Transporte | Pon tu precio - Didi
INFO:     127.0.0.1:45734 - "POST /analizar HTTP/1.1" 200 OK
Factura analizada en 7.16 s: 20000.0 | 2026-11-01 | Resta

## 6. Apagar
Cuando termine la demo.

In [ ]:
for p in ("uvicorn dai_servidor", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", p])
print("Dai apagado.")